# Model Tuning & Deployment

After building a baseline model, **hyperparameter tuning** squeezes out better performance.
Then **saving the model** lets you reuse it without retraining every time.

## What's Covered

| Topic | Tool | Purpose |
|---|---|---|
| **Grid Search** | `GridSearchCV` | Try every combination of hyperparameters exhaustively |
| **Random Search** | `RandomizedSearchCV` | Sample random combinations — faster for large grids |
| **Cross-Validation** | `cross_val_score`, `StratifiedKFold` | More reliable evaluation than a single train/test split |
| **Pipeline** | `Pipeline`, `ColumnTransformer` | Chain preprocessing + model into one reusable object |
| **Model Saving** | `joblib`, `pickle` | Persist trained model to disk, reload for inference later |

## What Are Hyperparameters?

- **Model weights** (e.g. linear regression slopes) → learned automatically from data by `.fit()`
- **Hyperparameters** (e.g. `max_depth`, `n_neighbors`, `C`) → **you must set them before training**
- Choosing the right hyperparameter values can significantly improve accuracy
- Finding them manually by trial-and-error is tedious → `GridSearchCV` / `RandomizedSearchCV` automates this

In [ ]:
## ── GRIDSEARCHCV — Exhaustive Hyperparameter Tuning ─────────────────────────

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score

np.random.seed(42)
X, y = make_classification(n_samples=400, n_features=6, n_informative=4,
                            n_redundant=1, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# param_grid = a dict of hyperparameter names → list of values to try
# GridSearchCV tries EVERY combination (3 × 3 × 2 = 18 combinations here)
# With cv=5 it trains 18 × 5 = 90 models total
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth':    [None, 5, 10],
    'min_samples_split': [2, 5],
}

grid_search = GridSearchCV(
    estimator  = RandomForestClassifier(random_state=42),
    param_grid = param_grid,
    cv         = 5,           # 5-fold cross-validation per combination
    scoring    = 'accuracy',  # what to optimise
    n_jobs     = -1,          # use all CPU cores in parallel
    verbose    = 1,
)
grid_search.fit(X_train, y_train)

print('=== GridSearchCV Results ===')
print(f'Total combinations tried: {len(grid_search.cv_results_["params"])}')
print(f'Best parameters  : {grid_search.best_params_}')
print(f'Best CV score    : {grid_search.best_score_:.4f}')
print(f'Test score       : {grid_search.score(X_test, y_test):.4f}')

# Top 5 combinations sorted by CV score
results_df = pd.DataFrame(grid_search.cv_results_)
top5 = results_df.sort_values('rank_test_score').head(5)
print('\n=== Top 5 Configurations ===')
cols = ['param_n_estimators', 'param_max_depth', 'param_min_samples_split',
        'mean_test_score', 'std_test_score', 'rank_test_score']
print(top5[cols].to_string(index=False))

# best_estimator_ is already fitted with best_params_ on the full training set
best_model = grid_search.best_estimator_
print(f'\nBaseline (default RF) test acc : {accuracy_score(y_test, RandomForestClassifier(random_state=42).fit(X_train, y_train).predict(X_test)):.4f}')
print(f'GridSearchCV best model test   : {accuracy_score(y_test, best_model.predict(X_test)):.4f}')

In [ ]:
## ── RANDOMIZEDSEARCHCV — Fast Approximate Tuning ────────────────────────────

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import accuracy_score
from scipy.stats import randint

np.random.seed(42)
X, y = __import__('sklearn.datasets', fromlist=['make_classification']).make_classification(
    n_samples=400, n_features=6, n_informative=4, n_redundant=1, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# RandomizedSearchCV samples n_iter random combinations instead of trying all
# Use scipy.stats distributions for continuous parameters → much larger search space
# This grid has 100 × 10 × 3 = 3000 combinations — GridSearch would take too long
param_dist = {
    'n_estimators':      randint(50, 300),     # sample integer between 50-299
    'max_depth':         [None, 3, 5, 7, 10, 15, 20],
    'min_samples_split': randint(2, 20),        # sample integer between 2-19
    'max_features':      ['sqrt', 'log2', None],
}

rand_search = RandomizedSearchCV(
    estimator          = RandomForestClassifier(random_state=42),
    param_distributions = param_dist,
    n_iter             = 30,    # try only 30 random combinations (vs 3000+ exhaustive)
    cv                 = 5,
    scoring            = 'accuracy',
    n_jobs             = -1,
    random_state       = 42,
    verbose            = 0,
)
rand_search.fit(X_train, y_train)

print('=== RandomizedSearchCV Results ===')
print(f'Combinations sampled : 30  (out of 3000+ possible)')
print(f'Best parameters      : {rand_search.best_params_}')
print(f'Best CV score        : {rand_search.best_score_:.4f}')
print(f'Test score           : {rand_search.score(X_test, y_test):.4f}')

print('\n=== GridSearch vs RandomizedSearch ===')
comparison = pd.DataFrame({
    'Property':        ['Combinations tried', 'Speed', 'Coverage', 'Best for'],
    'GridSearchCV':    ['ALL (exhaustive)', 'Slow for large grids', 'Complete',
                        'Small grids (< ~100 combos)'],
    'RandomizedSearchCV': ['n_iter (you choose)', 'Fast', 'Random sample',
                           'Large grids, continuous params'],
})
print(comparison.to_string(index=False))

In [ ]:
## ── PIPELINE + COLUMNTRANSFORMER ────────────────────────────────────────────

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report

np.random.seed(42)
n = 300

# Mixed dataset: numeric + categorical + missing values
df = pd.DataFrame({
    'Age':        np.where(np.random.rand(n) < 0.1, np.nan,
                           np.random.randint(18, 65, n).astype(float)),
    'Income':     np.where(np.random.rand(n) < 0.08, np.nan,
                           np.random.randint(20000, 100000, n).astype(float)),
    'StudyHours': np.random.uniform(1, 10, n),
    'City':       np.random.choice(['Delhi', 'Mumbai', 'Pune', 'Kolkata'], n),
    'Gender':     np.random.choice(['Male', 'Female'], n),
})
# Target: pass if age is younger + income higher + study hours high
df['Passed'] = (
    (100 - df['Age'].fillna(40)) * 0.3
    + df['Income'].fillna(50000) / 1000
    + df['StudyHours'] * 5
    + np.random.normal(0, 8, n) > 70
).astype(int)

X = df.drop('Passed', axis=1)
y = df['Passed']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ── ColumnTransformer: apply different steps to different column types ─────────
numeric_cols     = ['Age', 'Income', 'StudyHours']
categorical_cols = ['City', 'Gender']

# Numeric pipeline: impute missing → scale
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),  # fill NaN with median
    ('scaler',  StandardScaler()),                  # then scale
])

# Categorical pipeline: impute missing → one-hot encode
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),   # fill NaN with mode
    ('encoder', OneHotEncoder(handle_unknown='ignore')),    # then encode
])

# ColumnTransformer applies each pipeline to its designated columns simultaneously
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline,     numeric_cols),
    ('cat', categorical_pipeline, categorical_cols),
])

# Full Pipeline: preprocessing + model in one object
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42)),
])

# ── Why Pipeline prevents data leakage ────────────────────────────────────────
# When used with cross_val_score, the pipeline re-fits the preprocessor
# on each training fold separately — test fold statistics never contaminate training
# If you scaled BEFORE the pipeline, the scaler would learn from the test fold too
full_pipeline.fit(X_train, y_train)

print('=== Full Pipeline: Preprocessing + Model ===')
print(full_pipeline)

print(f'\nTest Accuracy : {accuracy_score(y_test, full_pipeline.predict(X_test)):.4f}')
print(classification_report(y_test, full_pipeline.predict(X_test)))

# 5-Fold CV on the full pipeline — preprocessing is correctly re-fit per fold
cv_scores = cross_val_score(full_pipeline, X, y, cv=5, scoring='accuracy')
print(f'5-Fold CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

# ── GridSearchCV with Pipeline ─────────────────────────────────────────────────
# Note: hyperparameter names use the format "stepname__paramname"
from sklearn.model_selection import GridSearchCV

param_grid = {
    'model__n_estimators': [50, 100],
    'model__max_depth':    [None, 5],
}

gs = GridSearchCV(full_pipeline, param_grid, cv=3, scoring='accuracy', n_jobs=-1)
gs.fit(X_train, y_train)
print(f'\nGridSearch best params : {gs.best_params_}')
print(f'GridSearch test score  : {gs.score(X_test, y_test):.4f}')

# ── Predict on raw unseen data — no manual preprocessing needed ───────────────
new_person = pd.DataFrame({
    'Age': [np.nan], 'Income': [45000.0], 'StudyHours': [7.0],
    'City': ['Mumbai'], 'Gender': ['Female'],
})
pred = gs.predict(new_person)
print(f'\nNew person prediction  : {"Pass" if pred[0] == 1 else "Fail"}')
print('→ Pipeline handled the NaN automatically — no manual imputation needed')

In [ ]:
## ── MODEL SAVING & LOADING ───────────────────────────────────────────────────

import joblib
import pickle
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import make_classification
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Train a simple pipeline to demonstrate saving and loading
X, y = make_classification(n_samples=300, n_features=4, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  DecisionTreeClassifier(max_depth=5, random_state=42)),
])
pipeline.fit(X_train, y_train)

original_acc = accuracy_score(y_test, pipeline.predict(X_test))
original_preds = pipeline.predict(X_test)
print(f'Trained pipeline accuracy: {original_acc:.4f}')

# ── SAVING WITH JOBLIB ─────────────────────────────────────────────────────────
# joblib is the RECOMMENDED way to save sklearn models
# Faster than pickle for objects that contain large numpy arrays (which sklearn models do)
# Supports optional compression: joblib.dump(model, path, compress=3)

path_joblib = 'student_model.joblib'
joblib.dump(pipeline, path_joblib)  # serialise pipeline to disk

file_size = os.path.getsize(path_joblib)
print(f'\n=== Saved with joblib ===')
print(f'File: {path_joblib}  ({file_size} bytes on disk)')

# Reload — creates an exact copy of the original Python object
pipeline_jl = joblib.load(path_joblib)
jl_acc = accuracy_score(y_test, pipeline_jl.predict(X_test))
preds_match = np.array_equal(original_preds, pipeline_jl.predict(X_test))
print(f'Reloaded accuracy : {jl_acc:.4f}  (same as original: {jl_acc == original_acc})')
print(f'Predictions match : {preds_match}  (every prediction is identical)')

# ── SAVING WITH PICKLE ─────────────────────────────────────────────────────────
# pickle is Python's built-in serialization — works for ANY Python object
# Slightly slower than joblib for large arrays, but perfectly fine for small models
# 'wb' = write binary mode (required for pickle)
# 'rb' = read binary mode

path_pickle = 'student_model.pkl'
with open(path_pickle, 'wb') as f:
    pickle.dump(pipeline, f)

file_size_pkl = os.path.getsize(path_pickle)
print(f'\n=== Saved with pickle ===')
print(f'File: {path_pickle}  ({file_size_pkl} bytes on disk)')

with open(path_pickle, 'rb') as f:
    pipeline_pkl = pickle.load(f)

pkl_acc = accuracy_score(y_test, pipeline_pkl.predict(X_test))
print(f'Reloaded accuracy : {pkl_acc:.4f}')

# ── JOBLIB VS PICKLE ───────────────────────────────────────────────────────────
print('\n=== joblib vs pickle ===')
comparison = pd.DataFrame({
    'Feature':         ['Speed (large models)', 'Compression', 'sklearn support',  'Best for'],
    'joblib':          ['Faster',              'Built-in',    'Recommended',       'sklearn pipelines / large arrays'],
    'pickle':          ['Slower',              'None',        'Works fine',        'Small models / any Python object'],
})
print(comparison.to_string(index=False))

print('\nRecommendation: use joblib for sklearn models, pickle for general Python objects')

# ── REAL-WORLD PATTERN ────────────────────────────────────────────────────────
print('''
=== Real-World Usage Pattern ===

── TRAINING SCRIPT (train_model.py) ──────────────────────────────────────────
  pipeline = Pipeline([('scaler', StandardScaler()), ('model', DecisionTreeClassifier())])
  pipeline.fit(X_train, y_train)
  joblib.dump(pipeline, 'model_v1.joblib')          # save to disk
  print('Model saved')

── INFERENCE / DEPLOYMENT SCRIPT (predict.py) ────────────────────────────────
  pipeline = joblib.load('model_v1.joblib')          # load ONCE at startup
  new_data  = pd.read_csv('new_students.csv')
  preds     = pipeline.predict(new_data)             # preprocessing is automatic
  print(preds)

Key point: because the Pipeline includes the scaler, the loaded model handles
raw (unscaled) data automatically — no manual preprocessing step needed.
''')

# Cleanup demo files
os.remove(path_joblib)
os.remove(path_pickle)
print('Demo files cleaned up.')

# ── PHASE 6 COMPLETE SUMMARY ──────────────────────────────────────────────────
print('''
=== Phase 6 Summary ===

GridSearchCV:
  → Tries every combination in param_grid (exhaustive)
  → Best for small grids (< ~100 combinations)
  → grid_search.best_params_   → winning hyperparameters
  → grid_search.best_estimator_ → already-fitted best model

RandomizedSearchCV:
  → Samples n_iter random combinations (approximate but fast)
  → Best for large grids or continuous distributions
  → Works the same API as GridSearchCV

StratifiedKFold + cross_val_score:
  → K-fold CV preserves class ratio in each fold
  → More reliable accuracy estimate than a single train/test split

Pipeline:
  → Pipeline([('step1', transformer), ('step2', model)])
  → .fit(X_train) fits every step; .predict(X_test) applies every step
  → Prevents data leakage, works with GridSearchCV via step__param syntax

ColumnTransformer:
  → Apply different pipelines to different columns simultaneously
  → preprocessor = ColumnTransformer([('num', num_pipe, num_cols),
                                       ('cat', cat_pipe, cat_cols)])

Model Saving:
  → joblib.dump(model, 'file.joblib')   /  joblib.load('file.joblib')
  → pickle.dump(model, open('file.pkl','wb'))  /  pickle.load(open('file.pkl','rb'))
''')